In [ ]:
# Individual LDA Analysis for Korean Stops Perception
# Model: response ~ svot + sf0 for each participant
# Using Linear Discriminant Analysis (LDA)

import pandas as pd
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import warnings
warnings.filterwarnings('ignore')

# Load preprocessed data
data = pd.read_csv('./output/perception_preprocessed_data.csv')

print(f"Data shape: {data.shape}")
print(f"\nColumns: {data.columns.tolist()}")
print(f"\nNumber of unique subjects: {data['subject'].nunique()}")
print(f"\nResponse distribution:\n{data['response'].value_counts()}")
data.head()

In [ ]:
# Fit LDA models for each subject and store for later prediction

import joblib

all_models = {}  # Dictionary to store fitted models
failed_subjects = []

subjects = sorted(data['subject'].unique())

for subject in subjects:
    # Get subject data
    subject_data = data[data['subject'] == subject].copy()
    
    # Check if subject has enough data and all response categories
    if len(subject_data) < 10:
        failed_subjects.append((subject, 'insufficient_data'))
        continue
    
    if subject_data['response'].nunique() < 3:
        failed_subjects.append((subject, 'insufficient_categories'))
        continue
    
    try:
        # Prepare features and target
        X = subject_data[['svot', 'sf0']].values
        y = subject_data['response'].values
        
        # Fit LDA model
        lda = LinearDiscriminantAnalysis()
        lda.fit(X, y)
        
        # Store the fitted model
        all_models[subject] = lda
        
    except Exception as e:
        failed_subjects.append((subject, str(e)))
        continue

print(f"\nSuccessfully fitted models: {len(subjects) - len(failed_subjects)}")
print(f"Failed models: {len(failed_subjects)}")

if len(failed_subjects) > 0:
    print("\nFailed subjects (first 10):")
    for subj, reason in failed_subjects[:10]:
        print(f"  Subject {subj}: {reason}")

print(f"\nTotal models fitted: {len(all_models)}")
print(f"Subject IDs: {list(all_models.keys())[:10]}...")  # Show first 10

In [ ]:
# Save models to file
import os

models_path = './output/individual_lda_models.joblib'

# Create output directory if it doesn't exist
os.makedirs('./output', exist_ok=True)

# Save all models to a single file
joblib.dump(all_models, models_path)
print(f"Models saved to: {models_path}")
print(f"Number of models saved: {len(all_models)}")

# Show example model info
if len(all_models) > 0:
    example_subject = list(all_models.keys())[0]
    example_model = all_models[example_subject]
    print(f"\nExample model (Subject {example_subject}):")
    print(f"  Classes: {example_model.classes_}")
    print(f"  Number of features: {example_model.scalings_.shape[0]}")
    print(f"  Number of discriminants: {example_model.scalings_.shape[1]}")

## 나중에 모델을 로드해서 예측하는 방법

다른 파일에서 저장된 모델을 로드하고 사용하는 예제:

In [ ]:
# 예제: 저장된 모델 로드하고 예측하기
# import joblib
# import pandas as pd
# import numpy as np

# # 모델들 로드
# loaded_models = joblib.load('./output/individual_lda_models.joblib')
# print(f"Loaded {len(loaded_models)} models")

# # 특정 subject의 모델 사용
# subject_id = list(loaded_models.keys())[0]  # 첫 번째 subject
# model = loaded_models[subject_id]

# # 새로운 데이터로 예측
# # 예: svot=20, sf0=0.5
# new_data = np.array([[20, 0.5]])
# prediction = model.predict(new_data)
# probabilities = model.predict_proba(new_data)

# print(f"\nSubject {subject_id}:")
# print(f"Prediction: {prediction[0]}")
# print(f"Probabilities: {probabilities[0]}")
# print(f"Classes: {model.classes_}")